In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import csv
import os
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import pickle


plt.rcParams['font.size'] = 20
plt.rcParams['axes.titlesize'] = 30
plt.rcParams['axes.labelsize'] = 25
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['legend.fontsize'] = 20


In [ ]:
color_schemes = [
    [ '#FFB2B2'],
    [ '#B2D9FF'],
    [ '#B2FFB2'],
    [ '#FFECB2'],
    [ '#E0B2FF'],
    [ '#B2FFFF'],
    [ '#B2C8DF'],
    [ '#DFA8A8'],
    [ '#B5D0C1'],
    [ '#F0C4A3'],
    [ '#C6BBD3'],
    [ '#C2C4C6']
]

color_schemes_line = [
    [ '#FF7F7F'],
    [ '#7FBFFF'],
    [ '#7fab87'],
    [ '#EDB611'],
    [ '#CC7FFF'],
    [ '#1EE8E8'],
    [ '#3A5F8E'],
    [ '#A63D40'],
    [ '#447D63'],
    [ '#D17A38'],
    [ '#6D558B'],
    [ '#64666A']
]

In [ ]:
np.random.seed(99)

In [ ]:
%run ./ESN.ipynb
%run ./Reconstruct.ipynb

---

In [ ]:
# load data

with open("./data/traffic_data_6_9.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    X = [row for row in reader][1:]
    X = np.array([list(map(float, row[1:])) for row in X])
    
data_length = X.shape[0]
x = np.linspace(0, data_length - 1, data_length)
int_mul = 5
x_new = np.linspace(0, data_length - 1, data_length * int_mul)



variables = np.array(['400911', '400863', '402364','404753','400001','400965','402362'])
seq = list(range(len(variables)))
N_nodes = len(seq)
X_new = np.zeros((data_length * int_mul, X.shape[1]))
i = 0
for item in seq:
    tmp = np.array(X[:,item])
    f = interp1d(x, tmp, kind = 'cubic')
    # f = interp1d(x, tmp, kind = 'linear')
    arr = f(x_new)
    
    X_new[:,i] = arr.T
    i += 1

X = X_new
    
q1 = np.percentile(X, 25, interpolation='midpoint', axis=0)
q2 = np.percentile(X, 50, interpolation='midpoint', axis=0)
q3 = np.percentile(X, 75, interpolation='midpoint', axis=0)

X = (X - q2)/(q3 - q1)

print(X.shape)

N_washout = 5 * int_mul
N_train = 1500 * int_mul
N_tstart = 5 * int_mul
N_test = 1500 * int_mul

X_washout = X[:N_washout]

X_t = X[N_washout:N_washout+N_train-1]
Y_t = X[N_washout+1:N_washout+N_train]

X_test = X[N_tstart:N_tstart + N_test]

raw_segment_length = 36 * int_mul
keep_last_r = 30 * int_mul

In [ ]:
window_length = 1

Array_to_plot = [[],[],[],[],[],[],[],[],[],[]]
labels=['IRC']

N_x = X.shape[1]


N_dim   =  1 * N_x # dimension of inputs (and outputs)


N_net = 20


T, D = X.shape  

k = 0   
TDI_window = []
N_rep, N_evo = 30, 5

offset = np.zeros([N_net, N_rep])
for s in range(N_net):
    for rep in range(N_rep):
        offset[s,rep] = np.random.randint(N_train - N_evo - N_washout  -1)

test_window = 10




X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

for i in range((test_window - 1) * window_length, T):
    segment = X[i - np.arange(test_window) * window_length].reshape(-1)
    X_segments[i - (test_window - 1) * window_length] = segment


N_washout_segments = N_washout
N_train_segments   = N_train
N_tstart_segments  = N_tstart
N_test_segments    = T-N_train_segments


N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
N_units_segments = 100 * N_dim_segments #units in the reservoir 

# washout
X_washout_segments= X_segments[:N_washout_segments]
# training
X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

X_test_segments = X_segments[N_tstart_segments:]

TDInet = np.zeros([N_net,test_window,N_x,N_x])






for net in range(N_net):
    esn = ESN_traffic(N_units_segments, N_dim_segments, N_dim_segments)
    esn.train(X_t_segments, raw_segment_length, keep_last_r)


#esn = ESN(N_units, N_dim, N_dim)
#esn.train(X_washout, X_t, Y_t)


    for T in range (0, test_window):
#TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
        TDInet[net, T] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 1, N_dim, N_rep, test_window, 0, 0,  N_washout, N_evo, offset[net], T)
#TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


TDI = np.mean(np.array(TDInet), axis=0)
#TDI_window.append(np.max(TDI)*(softmax_entropy_score(TDI)+ 1e-6))
for t in range (test_window):       
    TDI_window.append(np.max(TDI[t]))
    Array_to_plot [t].append(TDI[t])
#TDI_window.append(np.std(TDI)/(np.mean(TDI) + 1e-6))

#TDI_window.append(np.mean(TDI))
#TDI_window.append(np.max(TDI)/(np.mean(TDI) + 1e-6))
#TDI_window.append(gated_max_softmax_entropy_score(TDI))




plt.plot(TDI_window)
plt.xticks(ticks= range(len(TDI_window)), labels = [i for i in range(len(TDI_window))])

save_dict = {
    'TDI_window': TDI_window,
    'Array_to_plot': Array_to_plot,
    'TDInet': TDInet,
    'N_x':N_x,
    'test_window':test_window 
    }

save_dir = os.path.join('result', 'traffic_results')

os.makedirs(save_dir, exist_ok=True) 

file_path = os.path.join(save_dir, 'traffic_experiment_6_9.pkl')


with open(file_path, 'wb') as f:
    pickle.dump(save_dict, f)
    
print(f"data has been saved to: {file_path}")

    





In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'traffic_results', 'traffic_experiment_6_9.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window = loaded_data['TDI_window']
Array_to_plot = loaded_data['Array_to_plot']
TDInet = loaded_data['TDInet']
N_x = loaded_data['N_x']
test_window = loaded_data['test_window']

In [ ]:
save_dir = os.path.join('result', 'figure')
os.makedirs(save_dir, exist_ok=True)

variables = np.array(['400911', '400863', '402364','404753','400001','400965','402362'])
target_idx = 4  
node_name_target = '400001'

sensor_indices = [i for i in range(len(variables)) if i != target_idx]

def draw_violin(ax, data, title, fill_colors, line_colors):
    vp = ax.violinplot(
        data,
        positions=np.arange(1, test_window + 1),
        showmeans=True, showextrema=True, showmedians=False, bw_method=0.5
    )
    
    for i, pc in enumerate(vp['bodies']):
        color_idx = i % len(fill_colors)
        pc.set_facecolor(fill_colors[color_idx])
        pc.set_edgecolor('darkgray')
        pc.set_alpha(0.85)

    vp['cmeans'].set_color(line_colors)
    vp['cmeans'].set_linewidth(2.5)
    vp['cmaxes'].set_color(line_colors)
    vp['cmins'].set_color(line_colors)  
    vp['cbars'].set_color(line_colors)
    
    ax.set_xticks(np.arange(1, test_window + 1))
    ax.set_xticklabels([str(i) for i in range(test_window)])
    return ax

for plot_idx, sensor_idx in enumerate(sensor_indices):
    node_name_sensor = variables[sensor_idx]
    
    data_to_target = [TDInet[:, t, target_idx, sensor_idx] for t in range(test_window)]
    data_from_target = [TDInet[:, t, sensor_idx, target_idx] for t in range(test_window)]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    
    c_fill_left = color_schemes[plot_idx]
    c_line_left = color_schemes_line[plot_idx]
    draw_violin(axes[0], data_to_target, f"{node_name_sensor} to {node_name_target}", c_fill_left, c_line_left)

    c_fill_right = color_schemes[plot_idx + 6]
    c_line_right = color_schemes_line[plot_idx + 6]
    draw_violin(axes[1], data_from_target, f"{node_name_target} to {node_name_sensor}", c_fill_right, c_line_right)
    
    y_min = min(axes[0].get_ylim()[0], axes[1].get_ylim()[0])
    y_max = max(axes[0].get_ylim()[1], axes[1].get_ylim()[1])
    
    for ax in axes:
        ax.set_ylim(y_min, y_max)
        new_yticks = np.linspace(0, y_max, 4)
        ax.set_yticks(new_yticks)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    plt.tight_layout()
    
    filename = f"Panel_{plot_idx+1}_Bidirectional_{node_name_sensor}_vs_{node_name_target}_6_9"
    pathname = os.path.join(save_dir, f"{filename}.png")
    fig.savefig(pathname, bbox_inches='tight', dpi=600)

In [ ]:
# load data

with open("./data/traffic_data_9_12.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    X = [row for row in reader][1:]
    X = np.array([list(map(float, row[1:])) for row in X])
    
data_length = X.shape[0]
x = np.linspace(0, data_length - 1, data_length)
int_mul = 5
x_new = np.linspace(0, data_length - 1, data_length * int_mul)



variables = np.array(['400911', '400863', '402364','404753','400001','400965','402362'])
seq = list(range(len(variables)))
N_nodes = len(seq)
X_new = np.zeros((data_length * int_mul, X.shape[1]))
i = 0
for item in seq:
    tmp = np.array(X[:,item])
    f = interp1d(x, tmp, kind = 'cubic')
    # f = interp1d(x, tmp, kind = 'linear')
    arr = f(x_new)
    
    X_new[:,i] = arr.T
    i += 1

X = X_new
    
q1 = np.percentile(X, 25, interpolation='midpoint', axis=0)
q2 = np.percentile(X, 50, interpolation='midpoint', axis=0)
q3 = np.percentile(X, 75, interpolation='midpoint', axis=0)

X = (X - q2)/(q3 - q1)

print(X.shape)

N_washout = 5 * int_mul
N_train = 1500 * int_mul
N_tstart = 5 * int_mul
N_test = 1500 * int_mul

X_washout = X[:N_washout]

X_t = X[N_washout:N_washout+N_train-1]
Y_t = X[N_washout+1:N_washout+N_train]

X_test = X[N_tstart:N_tstart + N_test]

raw_segment_length = 36 * int_mul
keep_last_r = 30 * int_mul

In [ ]:
window_length = 1

Array_to_plot = [[],[],[],[],[],[],[],[],[],[]]
labels=['IRC']

N_x = X.shape[1]


N_dim   =  1 * N_x # dimension of inputs (and outputs)


N_net = 20


T, D = X.shape  

k = 0   
TDI_window = []
N_rep, N_evo = 30, 5

offset = np.zeros([N_net, N_rep])
for s in range(N_net):
    for rep in range(N_rep):
        offset[s,rep] = np.random.randint(N_train - N_evo - N_washout  -1)

test_window = 10




X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

for i in range((test_window - 1) * window_length, T):
    segment = X[i - np.arange(test_window) * window_length].reshape(-1)
    X_segments[i - (test_window - 1) * window_length] = segment


N_washout_segments = N_washout
N_train_segments   = N_train
N_tstart_segments  = N_tstart
N_test_segments    = T-N_train_segments


N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
N_units_segments = 100 * N_dim_segments #units in the reservoir 

# washout
X_washout_segments= X_segments[:N_washout_segments]
# training
X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

X_test_segments = X_segments[N_tstart_segments:]

TDInet = np.zeros([N_net,test_window,N_x,N_x])






for net in range(N_net):
    esn = ESN_traffic(N_units_segments, N_dim_segments, N_dim_segments)
    esn.train(X_t_segments, raw_segment_length, keep_last_r)


#esn = ESN(N_units, N_dim, N_dim)
#esn.train(X_washout, X_t, Y_t)


    for T in range (0, test_window):
#TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
        TDInet[net, T] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 1, N_dim, N_rep, test_window, 0, 0,  N_washout, N_evo, offset[net], T)
#TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


TDI = np.mean(np.array(TDInet), axis=0)
#TDI_window.append(np.max(TDI)*(softmax_entropy_score(TDI)+ 1e-6))
for t in range (test_window):       
    TDI_window.append(np.max(TDI[t]))
    Array_to_plot [t].append(TDI[t])
#TDI_window.append(np.std(TDI)/(np.mean(TDI) + 1e-6))

#TDI_window.append(np.mean(TDI))
#TDI_window.append(np.max(TDI)/(np.mean(TDI) + 1e-6))
#TDI_window.append(gated_max_softmax_entropy_score(TDI))




plt.plot(TDI_window)
plt.xticks(ticks= range(len(TDI_window)), labels = [i for i in range(len(TDI_window))])

save_dict = {
    'TDI_window': TDI_window,
    'Array_to_plot': Array_to_plot,
    'TDInet': TDInet,
    'N_x':N_x,
    'test_window':test_window 
    }

save_dir = os.path.join('result', 'traffic_results')

os.makedirs(save_dir, exist_ok=True) 

file_path = os.path.join(save_dir, 'traffic_experiment_9_12.pkl')


with open(file_path, 'wb') as f:
    pickle.dump(save_dict, f)
    
print(f"data has been saved to: {file_path}")

    





In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'traffic_results', 'traffic_experiment_9_12.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window = loaded_data['TDI_window']
Array_to_plot = loaded_data['Array_to_plot']
TDInet = loaded_data['TDInet']
N_x = loaded_data['N_x']
test_window = loaded_data['test_window']

In [ ]:
save_dir = os.path.join('result', 'figure')
os.makedirs(save_dir, exist_ok=True)

variables = np.array(['400911', '400863', '402364','404753','400001','400965','402362'])
target_idx = 4  
node_name_target = '400001'

sensor_indices = [i for i in range(len(variables)) if i != target_idx]

def draw_violin(ax, data, title, fill_colors, line_colors):
    vp = ax.violinplot(
        data,
        positions=np.arange(1, test_window + 1),
        showmeans=True, showextrema=True, showmedians=False, bw_method=0.5
    )
    
    for i, pc in enumerate(vp['bodies']):
        color_idx = i % len(fill_colors)
        pc.set_facecolor(fill_colors[color_idx])
        pc.set_edgecolor('darkgray')
        pc.set_alpha(0.85)

    vp['cmeans'].set_color(line_colors)
    vp['cmeans'].set_linewidth(2.5)
    vp['cmaxes'].set_color(line_colors)
    vp['cmins'].set_color(line_colors)  
    vp['cbars'].set_color(line_colors)
    
    ax.set_xticks(np.arange(1, test_window + 1))
    ax.set_xticklabels([str(i) for i in range(test_window)])
    return ax

for plot_idx, sensor_idx in enumerate(sensor_indices):
    node_name_sensor = variables[sensor_idx]
    
    data_to_target = [TDInet[:, t, target_idx, sensor_idx] for t in range(test_window)]
    data_from_target = [TDInet[:, t, sensor_idx, target_idx] for t in range(test_window)]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    
    c_fill_left = color_schemes[plot_idx]
    c_line_left = color_schemes_line[plot_idx]
    draw_violin(axes[0], data_to_target, f"{node_name_sensor} to {node_name_target}", c_fill_left, c_line_left)

    c_fill_right = color_schemes[plot_idx + 6]
    c_line_right = color_schemes_line[plot_idx + 6]
    draw_violin(axes[1], data_from_target, f"{node_name_target} to {node_name_sensor}", c_fill_right, c_line_right)
    
    y_min = min(axes[0].get_ylim()[0], axes[1].get_ylim()[0])
    y_max = max(axes[0].get_ylim()[1], axes[1].get_ylim()[1])
    
    for ax in axes:
        ax.set_ylim(y_min, y_max)
        new_yticks = np.linspace(0, y_max, 4)
        ax.set_yticks(new_yticks)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    plt.tight_layout()
    
    filename = f"Panel_{plot_idx+1}_Bidirectional_{node_name_sensor}_vs_{node_name_target}_9_12"
    pathname = os.path.join(save_dir, f"{filename}.png")
    fig.savefig(pathname, bbox_inches='tight', dpi=600)